In [2]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train = pd.read_csv("../data/X_train_clean.csv", index_col=0)
y_train = pd.read_csv("../data/y_train.csv", index_col=0).squeeze("columns")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
print(scores.round(4))
print(scores.mean(), scores.std())

[0.8532 0.8425 0.8504 0.8497 0.8551]
0.8501773896275232 0.0043126733823909075


In [3]:
from sklearn.model_selection import cross_validate

def cv_report(model, name):
    res = cross_validate(
        model, X_train, y_train, cv=cv,
        scoring=["roc_auc", "precision", "recall"]
    )
    print(name,
          "AUC:", res["test_roc_auc"].mean().round(4),
          "precision:", res["test_precision"].mean().round(3),
          "recall:", res["test_recall"].mean().round(3))

cv_report(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          "plain")
cv_report(make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=1000, class_weight="balanced")),
          "balanced")

plain AUC: 0.8502 precision: 0.594 recall: 0.175
balanced AUC: 0.8552 precision: 0.217 recall: 0.739


In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=50, n_jobs=-1, random_state=42
)
cv_report(rf, "random forest")

random forest AUC: 0.8638 precision: 0.62 recall: 0.123


In [5]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=15,
    min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, random_state=42, verbose=-1
)
cv_report(lgbm, "lightgbm default")

lightgbm default AUC: 0.8647 precision: 0.589 recall: 0.192


In [6]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [200, 300, 500],
    "learning_rate": [0.02, 0.05, 0.1],
    "num_leaves": [7, 15, 31],
    "min_child_samples": [20, 50, 100],
}

search = RandomizedSearchCV(
    LGBMClassifier(subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                   random_state=42, verbose=-1),
    param_dist, n_iter=15, cv=cv, scoring="roc_auc",
    random_state=42, n_jobs=-1
)
search.fit(X_train, y_train)

print(search.best_params_)
print(search.best_score_)

{'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 100, 'learning_rate': 0.02}
0.8651347069639064


In [7]:
from sklearn.metrics import roc_auc_score

X_test = pd.read_csv("../data/X_test_clean.csv", index_col=0)
y_test = pd.read_csv("../data/y_test.csv", index_col=0).squeeze("columns")

final = search.best_estimator_
final.fit(X_train, y_train)
proba_test = final.predict_proba(X_test)[:, 1]
print("final model test AUC:", roc_auc_score(y_test, proba_test))

# reference: the Week 1 logistic regression on the same test part
ref = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
ref.fit(X_train, y_train)
print("logistic test AUC:", roc_auc_score(y_test, ref.predict_proba(X_test)[:, 1]))

final model test AUC: 0.8684814931772196
logistic test AUC: 0.85741618448966


# 03_models: stronger models and a fair comparison

Goal: compare several models and choose a final one, using only the training data for decisions. The test set is used once, at the end.

Input: the cleaned training and test tables created by 02_baseline.ipynb (run that notebook first).

## Cross-validating the baseline

A single train/validation split gives one score that depends on which rows landed where. With 5-fold stratified cross-validation, the training data is cut into 5 parts that keep the 6.7% default rate. The model is trained on 4 parts and scored on the fifth, five times, and I average the AUC.

The scaler and the model are chained in a pipeline, so that inside each round the scaler learns only from the training parts and nothing leaks from the validation part.

Result: AUC 0.8502 on average, standard deviation 0.0043. The Week 1 test AUC (0.857) was within the normal range, slightly on the lucky side. From now on I compare models by the cross-validation mean, and the standard deviation tells me which differences are real: a gain smaller than about 0.004 is noise.

## Class weights

With only 6.7% defaults the model sees a defaulter 1 time in 15. With class_weight="balanced", each default counts about 14 times more in training.

Result: the AUC rises slightly (0.8502 -> 0.8552), but the main effect is on the 0.5 threshold: recall goes from 0.175 to 0.739 and precision falls from 0.594 to 0.217. The weights make the model flag many more borrowers. Precision and recall at a fixed threshold are not comparable between models, so AUC is used for the comparison.

## Random forest

A random forest builds many decision trees on random samples of rows and columns and averages them. It can capture combinations of features that logistic regression cannot, for example high utilization together with a past-due record. min_samples_leaf=50 prevents the trees from memorizing individual borrowers. No scaling is needed.

Result: AUC 0.8638, a gain of about 0.014 over the logistic regression, more than three times the spread, so a real improvement.

## Gradient boosting (LightGBM)

Gradient boosting builds small trees one after another, each correcting the errors of the previous ones. It is often more accurate than a forest but easier to overfit, so I keep the trees small, use a low learning rate, and use random subsamples of rows and columns.

Default settings: AUC 0.8647.

Tuning: a random search over 15 combinations of 4 settings, scored by the same 5-fold cross-validation. Best: 300 trees, 15 leaves, learning rate 0.02, at least 100 borrowers per group, AUC 0.8651. The gain over the default is 0.0004, far below the standard deviation, so the model is not sensitive to these settings. The best score of a search is also slightly optimistic, because it is the maximum of 15 noisy numbers.

## Comparison and final choice

All models were scored with the same folds (cross-validation AUC). The three tree models are within 0.0013 of each other, which is within noise, and all are about 0.015 above the plain logistic regression.

Final choice: tuned LightGBM. It is statistically tied with the random forest, so the tie is broken by training speed and simpler handling of settings. The logistic regression stays as the explainable reference model.

The final model is refit on the full training data and scored once on the test set: test AUC [fill in], against [fill in] for the logistic regression.
